In [7]:
x = np.linspace(0.40, 0.75, 351)
curves = []
for _, r in checks.iterrows():
    sd = (r["high"] - r["low"]) / (2 * 1.96)
    dens = np.exp(-0.5 * ((x - r["rate"]) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))
    curves.append(pd.DataFrame({"check": r["check"], "x": x, "density": dens}))

curves = pd.concat(curves)
curves.to_csv(out_dir / "curves.csv", index=False)
print(curves.shape)
curves.groupby("check")["density"].max().round(1)

(702, 3)


check
Neighbours within 0.1 s    35.2
Penalty-flipped pairs       8.5
Name: density, dtype: float64

In [6]:
long = pd.concat([
    checks.assign(point="rate", order=0, value=checks["rate"]),
    checks.assign(point="range", order=1, value=checks["low"]),
    checks.assign(point="range", order=2, value=checks["high"]),
])[["check", "point", "order", "value", "n"]]

long.to_csv(out_dir / "forest.csv", index=False)
long.round(4)

,check,point,order,value,n
0,Neighbours within 0.1 s,rate,0,0.5347,1943
1,Penalty-flipped pairs,rate,0,0.6019,108
0,Neighbours within 0.1 s,range,1,0.5126,1943
1,Penalty-flipped pairs,range,1,0.5095,108
0,Neighbours within 0.1 s,range,2,0.5569,1943
1,Penalty-flipped pairs,range,2,0.6942,108


In [5]:
def interval(p, n):
    me = 1.96 * np.sqrt(p * (1 - p) / n)
    return p - me, p + me


rows = []
for label, rate, n in [
    ("Neighbours within 0.1 s", nt["front_ahead"].mean(), len(nt)),
    ("Penalty-flipped pairs", fl["slot_ahead_won"].mean(), len(fl)),
]:
    lo, hi = interval(rate, n)
    rows.append({"check": label, "rate": rate, "low": lo, "high": hi, "n": n})

checks = pd.DataFrame(rows)
checks.to_csv(out_dir / "checks.csv", index=False)
checks.round(4)

,check,rate,low,high,n
0,Neighbours within 0.1 s,0.5347,0.5126,0.5569,1943
1,Penalty-flipped pairs,0.6019,0.5095,0.6942,108


In [4]:
pp = build_pairs(pairs, table)
pp.to_csv(out_dir / "pairs.csv", index=False)

chk = pd.read_csv(out_dir / "pairs.csv")
nt = chk[chk["near_tie"] == 1]
fl = nt[nt["flipped"] == 1]
print(chk.shape, len(nt))
print(round(nt["front_ahead"].mean(), 4))
print(len(fl), round(fl["slot_ahead_won"].mean(), 4))

(4861, 15) 1943
0.5347
108 0.6019


In [3]:
q = pd.read_sql(Path("../sql/04_qualifying_times.sql").read_text(), con)

q["group"] = np.where(q["q3_millis"].notna(), 3,
                      np.where(q["q2_millis"].notna(), 2, 1))
q["t"] = q["q3_millis"].fillna(q["q2_millis"]).fillna(q["q1_millis"])

q = q.sort_values(["race_id", "qpos"])
back = q.groupby("race_id")[["qpos", "group", "t", "driver_id"]].shift(-1)
pairs = q[["race_id", "year", "driver_id", "qpos", "group", "t"]].join(
    back.add_suffix("_back")
)

ok = (pairs["group"] == pairs["group_back"]) & (pairs["qpos_back"] == pairs["qpos"] + 1)
pairs = pairs[ok].copy()
pairs["gap_ms"] = pairs["t_back"] - pairs["t"]
print(pairs["race_id"].nunique(), len(pairs))

393 6966


In [2]:
raw = pd.read_sql(Path("../sql/01_race_results.sql").read_text(), con)
chron = pd.read_sql("SELECT * FROM constructor_chronology", con)
table = build_table(raw, chron)
print(table.shape)

(8636, 21)


In [1]:
import sys
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path("../src").resolve()))

from clean import build_table
from export import build_pairs

con = sqlite3.connect("../data/raw/f1db.db")
out_dir = Path("../data/processed")